**CheXpert Plus cache (CPU, paper 2).** Needs the `REDIVIS_API_TOKEN` secret attached (Add-ons → Secrets) and Internet on.

Writes to /kaggle/working:
- `chexpert384.zip`: 384 px grayscale PNGs, named by their full path (`train/patientX/studyY/view1_frontal.png`). It holds every frontal **valid** image, plus every frontal **train** image of a random 25% of train patients (seed 0). The sample size was fixed from download speed alone, before any model was evaluated on CheXpert.
- `chexpert_meta.csv`: metadata without report text.
- `chexpert_labels_impression.csv`: CheXbert labels from the impression section, the original CheXpert convention.
- `chexlocalize_files.csv`: the file list of the CheXlocalize table, used to find the radiologist-labelled valid/test sets.

A time guard stops downloading at 10.5 h and keeps everything fetched so far.

In [ ]:
import os, sys, subprocess, time, json, zipfile, random
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "redivis"], check=True)
from kaggle_secrets import UserSecretsClient
os.environ["REDIVIS_API_TOKEN"] = UserSecretsClient().get_secret("REDIVIS_API_TOKEN")
import redivis, cv2, numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
T0 = time.time(); STOP_H = 10.5; W = "/kaggle/working"
ds = next(d for d in redivis.organization("AIMI").list_datasets() if d.name == "CheXpert Plus")
tables = {t.name: t for t in ds.list_tables()}

In [ ]:
# Metadata (no report text) and impression labels
meta = tables["df_chexpert_plus_240401"].to_pandas_dataframe()
keep = ["path_to_image", "frontal_lateral", "ap_pa", "deid_patient_id", "patient_report_date_order", "age", "sex",
        "race", "ethnicity", "interpreter_needed", "insurance_type", "recent_bmi", "deceased", "split"]
meta = meta[keep]
meta.to_csv(f"{W}/chexpert_meta.csv", index=False)
d = tables["CheXpert Labels"].to_directory().download(f"{W}/_labels", overwrite=True) if hasattr(tables["CheXpert Labels"], "to_directory") else tables["CheXpert Labels"].download_files(path=f"{W}/_labels")
lab_path = next(os.path.join(r, f) for r, _, fs in os.walk(f"{W}/_labels") for f in fs if f.startswith("impression"))
lab = pd.read_json(lab_path, lines=True)
lab.to_csv(f"{W}/chexpert_labels_impression.csv", index=False)
print(meta.shape, lab.shape, "label values:", sorted(pd.unique(lab.drop(columns="path_to_image").values.ravel().astype(float)), key=lambda v: (np.isnan(v), v)))

In [ ]:
# Which images: all frontal valid + frontal train of a random 25% of train patients (seed 0)
fr = meta[meta.frontal_lateral == "Frontal"]
pts = sorted(fr.loc[fr.split == "train", "deid_patient_id"].unique())
random.Random(0).shuffle(pts)
chosen = set(pts[: int(round(0.25 * len(pts)))])
want = set(fr.loc[(fr.split == "valid") | fr.deid_patient_id.isin(chosen), "path_to_image"].str.replace(".jpg", "", regex=False))
print(f"train patients {len(pts)}, sampled {len(chosen)}; images wanted {len(want)}")

In [ ]:
# File index -> full relative path. Redivis file names are not unique, and the PNG_train / PNG_valid
# paths have no split prefix (e.g. "patientXXXXX/study1/view1_frontal.png"), so the key is
# split prefix (from the table name) + the file's path without its extension.
def relkey(f, split):
    v = str(getattr(f, "path", "") or "").lstrip("/")
    if not v:
        return None
    for root in ("train/", "valid/"):
        i = v.find(root)
        if i >= 0:
            v = v[i + len(root):]
    return split + "/" + v.rsplit(".", 1)[0]
keyed = {}
for tname, split in (("PNG_valid", "valid"), ("PNG_train", "train")):
    t0 = time.time(); fl = tables[tname].list_files()
    print(tname, len(fl), f"listed in {time.time()-t0:.0f}s; example key:", relkey(fl[0], split))
    for f in fl:
        k = relkey(f, split)
        if k in want:
            keyed[k] = f
print(f"matched {len(keyed)} of {len(want)} wanted images; example wanted:", sorted(want)[0])
assert len(keyed) > 0.95 * len(want), "path mapping failed; compare the example key and example wanted above"


In [ ]:
# Download, resize to 384 (area), write one zip. 16-bit PNGs are scaled by their own max.
def to384(b):
    a = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_UNCHANGED)
    if a.ndim == 3:
        a = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    if a.dtype != np.uint8:
        a = (a.astype(np.float32) * (255.0 / max(float(a.max()), 1.0))).astype(np.uint8)
    a = cv2.resize(a, (384, 384), interpolation=cv2.INTER_AREA)
    return cv2.imencode(".png", a, [cv2.IMWRITE_PNG_COMPRESSION, 3])[1].tobytes(), a.dtype.name
def job(item):
    k, f = item
    for attempt in range(3):
        try:
            return k, to384(f.read())[0]
        except Exception as e:
            err = e; time.sleep(2 * (attempt + 1))
    return k, None
items = sorted(keyed.items()); done, failed = 0, []
with zipfile.ZipFile(f"{W}/chexpert384.zip", "w", zipfile.ZIP_STORED, allowZip64=True) as zf, ThreadPoolExecutor(48) as ex:
    for start in range(0, len(items), 2000):
        if (time.time() - T0) / 3600 > STOP_H:
            print("time guard: stopping at", done); break
        for k, data in ex.map(job, items[start:start + 2000]):
            if data is None: failed.append(k)
            else: zf.writestr(k + ".png", data); done += 1
        el = time.time() - T0
        print(f"{done}/{len(items)} done, {len(failed)} failed, {done/el:.1f} img/s overall, {el/3600:.2f} h", flush=True)
pd.Series(failed, name="failed").to_csv(f"{W}/failed.csv", index=False)
print("cached", done, "failed", len(failed))

In [ ]:
# CheXlocalize: list files only (to find the radiologist-labelled valid/test sets)
try:
    loc = next(d for d in redivis.organization("AIMI").list_datasets() if d.name == "CheXlocalize")
    lt = loc.list_tables()[0]
    lf = lt.list_files()
    pd.DataFrame({"name": [f.name for f in lf], "path": [str(getattr(f, "path", "")) for f in lf],
                  "size": [getattr(f, "size", None) for f in lf]}).to_csv(f"{W}/chexlocalize_files.csv", index=False)
    print(len(lf), "CheXlocalize files; non-image examples:", [str(getattr(f, "path", f.name)) for f in lf if not f.name.lower().endswith((".png", ".jpg"))][:40])
except Exception as e:
    print("CheXlocalize listing failed:", type(e).__name__, str(e)[:300])
import shutil; shutil.rmtree(f"{W}/_labels", ignore_errors=True)
print(os.listdir(W))